In [1]:
import pyspark
from pyspark.sql import SparkSession
from pathlib import Path

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/12 22:06:05 WARN Utils: Your hostname, Adrians-M2-Macbook.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.97 instead (on interface en0)
26/03/12 22:06:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/12 22:06:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
BASE_DIR=Path.cwd().parent
DATA_DIR=BASE_DIR/"data"
RAW_DATA_DIR=DATA_DIR/"raw"
PARQUET_DIR=DATA_DIR/"parquet"

In [4]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz -P {RAW_DATA_DIR}

--2026-03-12 22:06:19--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-13T06%3A04%3A50Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-13T05%3A04%3A41Z&ske=2026-03-13T06%3A04%3A50Z&sks=b&skv=2018-11-09&sig=CZypZZkM241SAh9DIqGbCiTiyVjtILGbAHYDQitVkFM%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MzM4MTk3OSwibmJmIjoxNzczMzc4Mzc5LCJwYXRoIjoi

In [5]:
!ls -lh {RAW_DATA_DIR}

total 524944
-rw-r--r--@ 1 acepeda  staff   124M Jul 14  2022 fhvhv_tripdata_2021-01.csv.gz
-rw-r--r--@ 1 acepeda  staff   124M Jul 14  2022 fhvhv_tripdata_2021-01.csv.gz.1
-rw-r--r--@ 1 acepeda  staff    12K Feb 22  2024 taxi_zone_lookup.csv


In [12]:
!gzip -dc {RAW_DATA_DIR}/fhvhv_tripdata_2021-01.csv.gz > {RAW_DATA_DIR}/fhvhv_tripdata_2021-01.csv

In [13]:
!wc -l {RAW_DATA_DIR}/fhvhv_tripdata_2021-01.csv

 11908469 /Users/acepeda/Documents/GitHub/zoomcamp-de/06-batch/data/raw/fhvhv_tripdata_2021-01.csv


In [40]:
df = spark.read \
    .option("header", "true")  \
    .csv(f'{RAW_DATA_DIR}/fhvhv_tripdata_2021-01.csv')
df.printSchema()
df.show()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- SR_Flag: string (nullable = true)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|   

In [19]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [20]:
!head -n 1001 {RAW_DATA_DIR}/fhvhv_tripdata_2021-01.csv > head.csv

In [22]:
import pandas as pd

In [23]:
df_pandas = pd.read_csv('head.csv')

In [24]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [25]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [27]:
from pyspark.sql import types

In [38]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True), 
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('SR_Flag', types.StringType(), True)
])

In [39]:
df = spark.read \
    .schema(schema) \
    .option("header", "true") \
    .csv(f'{RAW_DATA_DIR}/fhvhv_tripdata_2021-01.csv')
df.printSchema()
df.show()


root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|        

In [ ]:
# Lazy execution - doesn't execute until the action is called
df = df.repartition(24)

In [42]:

df.write.parquet(f'{PARQUET_DIR}/fhvhv/2021/01/', mode='overwrite')


26/03/12 23:26:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/12 23:26:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/12 23:26:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/12 23:26:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/12 23:26:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/12 23:26:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/12 23:26:54 WARN MemoryManager: Total allocation exceeds 95.

In [43]:
df = spark.read.parquet(f'{PARQUET_DIR}/fhvhv/2021/01/')
df.printSchema()
df.show()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- SR_Flag: string (nullable = true)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02880|2021-01-02 10:59:19|2021-01-02 11:09:35|         164|          90|   NULL|
|           HV0003|              B02883|2021-01-01 16:58:08|2021-01-01 17:05:04|         189|         106|   NULL|
|           HV0003|              B02872|2021-01-01 19:20:38|2021-01-01 19:25:25|          32|   

In [44]:
from pyspark.sql import functions as F

In [55]:
df.show()
df.count()


+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02880|2021-01-02 10:59:19|2021-01-02 11:09:35|         164|          90|   NULL|
|           HV0003|              B02883|2021-01-01 16:58:08|2021-01-01 17:05:04|         189|         106|   NULL|
|           HV0003|              B02872|2021-01-01 19:20:38|2021-01-01 19:25:25|          32|         185|   NULL|
|           HV0005|              B02510|2021-01-02 21:17:59|2021-01-02 21:41:44|          76|         210|   NULL|
|           HV0003|              B02869|2021-01-03 14:41:30|2021-01-03 14:49:41|         233|         170|   NULL|
|           HV0003|              B02875|2021-01-01 09:29:39|2021-01-01 09:51:18|

11908468

In [59]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'{num:03x}'


In [60]:
crazy_stuff('B02884')

's/b44'

In [61]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())
 

In [63]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

df.printSchema()


+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  a/b40| 2021-01-02|  2021-01-02|         164|          90|
|  a/b43| 2021-01-01|  2021-01-01|         189|         106|
|    b38| 2021-01-01|  2021-01-01|          32|         185|
|    9ce| 2021-01-02|  2021-01-02|          76|         210|
|    b35| 2021-01-03|  2021-01-03|         233|         170|
|    b3b| 2021-01-01|  2021-01-01|         256|          85|
|    9ce| 2021-01-03|  2021-01-03|         127|         236|
|  a/a7a| 2021-01-03|  2021-01-03|         121|          98|
|    b3e| 2021-01-01|  2021-01-01|         254|         254|
|    9ce| 2021-01-02|  2021-01-02|          60|         168|
|  s/b44| 2021-01-03|  2021-01-03|         226|          82|
|  s/b13| 2021-01-03|  2021-01-03|         215|         130|
|    b35| 2021-01-01|  2021-01-01|         114|         140|
|    9ce| 2021-01-02|  2

In [67]:
df  \
    .select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-02 10:59:19|2021-01-02 11:09:35|         164|          90|
|2021-01-01 16:58:08|2021-01-01 17:05:04|         189|         106|
|2021-01-01 19:20:38|2021-01-01 19:25:25|          32|         185|
|2021-01-03 14:41:30|2021-01-03 14:49:41|         233|         170|
|2021-01-01 09:29:39|2021-01-01 09:51:18|         256|          85|
|2021-01-03 16:36:33|2021-01-03 16:47:39|         121|          98|
|2021-01-01 23:16:09|2021-01-01 23:23:29|         254|         254|
|2021-01-03 18:30:49|2021-01-03 18:43:11|         226|          82|
|2021-01-03 16:06:43|2021-01-03 16:14:08|         215|         130|
|2021-01-01 19:37:32|2021-01-01 19:53:30|         114|         140|
|2021-01-01 02:38:49|2021-01-01 02:44:49|          75|          74|
|2021-01-02 09:21:59|2021-01-02 09:39:21|       

In [65]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,
